# 24.4 Differential evolution

Differential evolution mutates by adding scaled population differences to existing candidates.

DE learns step scale from the population itself. A difference vector becomes a mutation direction, crossover mixes the mutant into a target, and greedy replacement keeps improvements under a fixed black-box budget.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 244
rng = np.random.default_rng(SEED)


def sphere_loss(points, target=None):
    points = np.asarray(points, dtype=float)
    if target is None:
        target = np.zeros(points.shape[1])
    return np.sum(np.square(points - target), axis=1)


def rastrigin_loss(points):
    points = np.asarray(points, dtype=float)
    dim = points.shape[1]
    return 10.0 * dim + np.sum(np.square(points) - 10.0 * np.cos(2.0 * np.pi * points), axis=1)


def constrained_loss(points):
    raw = np.sum(np.square(points - np.array([1.0, -1.0])), axis=1)
    violation = np.maximum(0.0, np.sum(points, axis=1) - 0.25)
    return raw + 40.0 * np.square(violation)


def make_de_ladder():
    return [
        {"name": "D1 1-D known optimum", "dim": 1, "bounds": (-5.0, 5.0), "loss": lambda x: np.square(x[:, 0] - 3.0)},
        {"name": "D2 2-D sphere", "dim": 2, "bounds": (-5.0, 5.0), "loss": lambda x: sphere_loss(x, target=np.array([2.0, 1.0]))},
        {"name": "D3 2-D Rastrigin", "dim": 2, "bounds": (-5.12, 5.12), "loss": rastrigin_loss},
        {"name": "D4 constrained bounds penalty", "dim": 2, "bounds": (-4.0, 4.0), "loss": constrained_loss},
        {"name": "D5 30-D Rastrigin", "dim": 30, "bounds": (-5.12, 5.12), "loss": rastrigin_loss},
    ]


def differential_trial(population, target_index, differential_weight, crossover_rate, bounds, rng):
    population_size, dim = population.shape
    choices = [index for index in range(population_size) if index != target_index]
    a_index, b_index, c_index = rng.choice(choices, size=3, replace=False)
    a = population[int(a_index)]
    b = population[int(b_index)]
    c = population[int(c_index)]
    mutant = a + differential_weight * (b - c)
    lower, upper = bounds
    mutant = np.clip(mutant, lower, upper)
    mask = rng.random(dim) < crossover_rate
    forced = int(rng.integers(0, dim))
    mask[forced] = True
    trial = np.where(mask, mutant, population[target_index])
    return trial, mutant, (a_index, b_index, c_index)


def differential_evolution(loss, dim, bounds, population_size=50, generations=55, differential_weight=0.7, crossover_rate=0.8, rng=None):
    if rng is None:
        rng = np.random.default_rng(SEED)
    lower, upper = bounds
    population = rng.uniform(lower, upper, size=(population_size, dim))
    values = loss(population)
    history = []
    last_trials = []
    for generation in range(generations):
        last_trials = []
        for index in range(population_size):
            trial, mutant, _ = differential_trial(population, index, differential_weight, crossover_rate, bounds, rng)
            trial_value = float(loss(trial.reshape(1, -1))[0])
            if trial_value <= values[index]:
                population[index] = trial
                values[index] = trial_value
            last_trials.append(trial)
        history.append(float(np.min(values)))
    best_index = int(np.argmin(values))
    return {"best": population[best_index], "best_loss": float(values[best_index]), "history": np.array(history), "population": population, "trials": np.array(last_trials)}


## The concept, built once (D1): mutate from differences

The lesson mutation is

$$v=a+F(b-c),\qquad u_j=\begin{cases}v_j,&r_j\lt CR\\x_{i,j},&r_j\ge CR.\end{cases}$$

With $a=(1,1)$, $b=(3,0)$, $c=(0,2)$, and $F=0.5$, the mutant must be $(2.5,0)$.

In [ ]:

a = np.array([1.0, 1.0])
b = np.array([3.0, 0.0])
c = np.array([0.0, 2.0])
mutant = a + 0.5 * (b - c)

print("mutant:", mutant)

assert np.allclose(mutant, np.array([2.5, 0.0]))


## Greedy replacement

For minimization, replace the target if $f(u)\le f(x_i)$. The lesson compares old $x=(0,0)$ to trial $u=(2.5,0)$ for target $(2,1)$: $5.0$ versus $1.25$, so replacement happens.

In [ ]:

old = np.array([[0.0, 0.0]])
trial = np.array([[2.5, 0.0]])
target = np.array([2.0, 1.0])
old_loss = float(sphere_loss(old, target=target)[0])
trial_loss = float(sphere_loss(trial, target=target)[0])
replace = trial_loss < old_loss

print("old loss", old_loss)
print("trial loss", trial_loss)
print("replace", replace)

assert np.isclose(old_loss, 5.0)
assert np.isclose(trial_loss, 1.25)
assert replace


## The dataset ladder

The inline F4 objective ladder uses a known 1-D optimum, a 2-D sphere, multimodal Rastrigin, a constrained penalty rung, and a 30-D Rastrigin stress test.

In [ ]:

ladder = make_de_ladder()

for rung in ladder:
    preview_rng = np.random.default_rng(SEED + rung["dim"])
    lower, upper = rung["bounds"]
    sample = preview_rng.uniform(lower, upper, size=(4, rung["dim"]))
    values = rung["loss"](sample)
    print(rung["name"], "shape", sample.shape, "sample loss", np.round(values, 3))


In [ ]:

results = []

for index, rung in enumerate(ladder):
    result = differential_evolution(rung["loss"], rung["dim"], rung["bounds"], population_size=50, generations=55, differential_weight=0.7, crossover_rate=0.8, rng=np.random.default_rng(SEED + index))
    results.append(result)
    print(f"{rung['name']}: best_loss={result['best_loss']:.4f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 6))

for index, rung in enumerate(ladder):
    ax = axes[0, index]
    lower, upper = rung["bounds"]
    if rung["dim"] == 1:
        grid = np.linspace(lower, upper, 200)
        ax.plot(grid, rung["loss"](grid.reshape(-1, 1)))
        ax.scatter(results[index]["population"][:, 0], rung["loss"](results[index]["population"]), s=12)
    else:
        grid_x = np.linspace(lower, upper, 80)
        grid_y = np.linspace(lower, upper, 80)
        mesh_x, mesh_y = np.meshgrid(grid_x, grid_y)
        points = np.column_stack([mesh_x.ravel(), mesh_y.ravel()])
        if rung["dim"] > 2:
            padded = np.zeros((points.shape[0], rung["dim"]))
            padded[:, :2] = points
            values = rung["loss"](padded)
            population = results[index]["population"][:, :2]
            trials = results[index]["trials"][:, :2]
        else:
            values = rung["loss"](points)
            population = results[index]["population"]
            trials = results[index]["trials"]
        ax.contourf(mesh_x, mesh_y, values.reshape(mesh_x.shape), levels=20, cmap="viridis_r")
        ax.scatter(population[:, 0], population[:, 1], c="white", s=10, edgecolor="black")
        for row in range(min(8, population.shape[0])):
            ax.arrow(population[row, 0], population[row, 1], trials[row, 0] - population[row, 0], trials[row, 1] - population[row, 1], color="cyan", alpha=0.5, head_width=0.08)
    ax.set_title(rung["name"].split()[0])

for index, result in enumerate(results):
    axes[1, index].plot(result["history"])
    axes[1, index].set_title("best loss")
    axes[1, index].set_xlabel("generation")

fig.suptitle("Population/trial panels and best-loss curves")
plt.tight_layout()


## Pitfall on D5: population too small

DE needs distinct, diverse vectors. A tiny population makes $b-c$ uninformative; the fix uses a larger population and preserves a diversity floor through normal DE sampling.

In [ ]:

d5 = ladder[-1]

tiny = differential_evolution(d5["loss"], d5["dim"], d5["bounds"], population_size=8, generations=55, differential_weight=0.5, crossover_rate=0.8, rng=np.random.default_rng(888))
larger = differential_evolution(d5["loss"], d5["dim"], d5["bounds"], population_size=60, generations=55, differential_weight=0.7, crossover_rate=0.8, rng=np.random.default_rng(888))

tiny_diversity = float(np.mean(np.std(tiny["population"], axis=0)))
larger_diversity = float(np.mean(np.std(larger["population"], axis=0)))

print("tiny best loss", tiny["best_loss"], "diversity", tiny_diversity)
print("fixed best loss", larger["best_loss"], "diversity", larger_diversity)


## Evaluate it + Practice

- Compare the reported best loss against a seeded random-search baseline with the same evaluation budget.
- Sanity check: D1 must hit the lesson's worked calculation before trusting D2–D5.
- Ablation: reduce population size below the number needed for reliable difference vectors should make the hardest rung worse or less stable.
- Failure signals: population variance collapses too early, bounds are violated, or the summary curve improves only by one lucky sample.
- Re-run with another seed only after the structural checks pass; do not tune from a single trace.

Practice prompts:
1. Change the hardest rung budget and explain whether convergence improves per objective call.
2. Add one diagnostic for diversity and plot it next to the metric.
3. Swap the D3 multimodal objective and predict which operator needs retuning.